# MNIST Knowledge Distillation (Organized Notebook)

This notebook is organized in sections:
1. Setup and hyperparameters
2. Data loading and full-batch tensors
3. Model and utility functions
4. Teacher/student training functions
5. Experiment execution (W&B + checkpoints)
6. Visualization of results

In [ ]:
import copy
import random
import subprocess
import sys
from datetime import datetime
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
else:
    print("Running outside Colab")

# Optional: persist artifacts/checkpoints in Google Drive when on Colab.
USE_GOOGLE_DRIVE = False

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    base_dir = Path("/content/drive/MyDrive/mo434_kd")
elif IN_COLAB:
    base_dir = Path("/content")
else:
    base_dir = Path(".")

base_dir.mkdir(parents=True, exist_ok=True)
data_dir = base_dir / "data"
checkpoints_dir = base_dir / "checkpoints"
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# Install missing packages automatically when running on Colab.
def ensure_package(pkg):
    try:
        __import__(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

if IN_COLAB:
    ensure_package("wandb")

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import datasets, transforms
import wandb

# Reproducibility
seed = 42
random.seed(seed)
torch.manual_seed(seed)

# Force GPU usage for all training and tensor batches.
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not available. In Colab, set Runtime > Change runtime type > Hardware accelerator = GPU, then rerun."
    )

device = torch.device("cuda")
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark = True

print(f"Using device: {device}")
print(f"CUDA device: {torch.cuda.get_device_name(0)}")

# W&B configuration
# Set entity to your team/user if needed, e.g. "my-team". Use None to keep default account.
WANDB_ENTITY = None
WANDB_PROJECT = "mo434-knowledge-distillation"
WANDB_GROUP = f"mnist-kd-fullbatch-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Optional: force explicit login in Colab (set to True if needed).
WANDB_LOGIN_IN_COLAB = False
if IN_COLAB and WANDB_LOGIN_IN_COLAB:
    wandb.login()

# Hyperparameters
teacher_epochs = 15
student_epochs = 20
teacher_lr = 1e-3
student_lr = 1e-3
lambda_kd = 0.5
temperatures = [1.0, 10.0]

print(f"Data directory: {data_dir.resolve()}")
print(f"Checkpoint directory: {checkpoints_dir.resolve()}")

Running outside Colab
Using device: cpu
Data directory: /Users/silvs/Documents/estudos/unicamp/Deep Learning/experiments-MO434/MO434/data
Checkpoint directory: /Users/silvs/Documents/estudos/unicamp/Deep Learning/experiments-MO434/MO434/checkpoints


## Runtime Configuration Check

The setup was executed in the previous code cell.
This cell confirms key paths and W&B project settings before training.

In [ ]:
print("Environment summary")
print(f"IN_COLAB: {IN_COLAB}")
print(f"Base directory: {base_dir}")
print(f"Data directory: {data_dir}")
print(f"Checkpoint directory: {checkpoints_dir}")
print(f"W&B project: {WANDB_PROJECT}")
print(f"W&B group: {WANDB_GROUP}")
print(f"Device: {device}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

Environment summary
IN_COLAB: False
Base directory: .
Data directory: data
Checkpoint directory: checkpoints
W&B project: mo434-knowledge-distillation
W&B group: mnist-kd-fullbatch-20260601-232038


## Data Loading and Full-Batch Preparation

MNIST is split into training and validation. Each split is converted into a single full batch tensor so each epoch performs one optimization step over the entire split.

In [3]:
# Data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

full_train = datasets.MNIST(root=str(data_dir), train=True, download=True, transform=transform)

# Validation split (fixed)
val_size = 5000
train_size = len(full_train) - val_size
train_dataset, val_dataset = torch.utils.data.random_split(
    full_train,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(seed),
)


def to_full_batch(dataset, device):
    # Full-batch training: one optimization step per epoch using all samples.
    x = torch.stack([dataset[i][0] for i in range(len(dataset))]).to(device)
    y = torch.tensor([dataset[i][1] for i in range(len(dataset))], dtype=torch.long, device=device)
    return x, y


x_train, y_train = to_full_batch(train_dataset, device)
x_val, y_val = to_full_batch(val_dataset, device)

print(f"Train batch shape: {x_train.shape}, Validation batch shape: {x_val.shape}")

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST/raw/train-images-idx3-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST/raw/train-labels-idx1-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST/raw/t10k-images-idx3-ubyte.gz to data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting data/MNIST/raw/t10k-labels-idx1-ubyte.gz to data/MNIST/raw

Train batch shape: torch.Size([55000, 1, 28, 28]), Validation batch shape: torch.Size([5000, 1, 28, 28])


## Model and Utility Functions

This section defines checkpoint saving, W&B dataset artifact logging, the MLP architecture, and evaluation/loss helpers.

In [4]:
def save_local_checkpoint(path, model, optimizer, epoch, best_val_acc, metadata):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch,
        "best_val_acc": best_val_acc,
        "metadata": metadata,
    }
    torch.save(checkpoint, path)


def log_dataset_artifact():
    dataset_run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        job_type="dataset",
        name="mnist-dataset-artifact",
        config={
            "dataset": "MNIST",
            "train_total": len(full_train),
            "train_split": train_size,
            "val_split": val_size,
            "normalization_mean": 0.1307,
            "normalization_std": 0.3081,
            "seed": seed,
            "full_batch": True,
            "created_at": datetime.now().isoformat(),
            "in_colab": IN_COLAB,
            "base_dir": str(base_dir),
        },
    )

    dataset_artifact = wandb.Artifact(
        name=f"mnist-full-batch-{seed}",
        type="dataset",
        description="MNIST dataset metadata and local files for KD experiment",
        metadata={
            "train_total": len(full_train),
            "train_split": train_size,
            "val_split": val_size,
            "seed": seed,
            "full_batch": True,
            "in_colab": IN_COLAB,
        },
    )

    data_root = data_dir / "MNIST"
    if data_root.exists():
        dataset_artifact.add_dir(str(data_root), name="MNIST")

    # Log a few dataset samples as metadata table.
    sample_table = wandb.Table(columns=["image", "label"])
    raw_images = full_train.dataset.data if hasattr(full_train, "dataset") else None
    raw_labels = full_train.dataset.targets if hasattr(full_train, "dataset") else None
    if raw_images is None:
        # random_split returns subsets, so dataset data lives in full_train.data.
        raw_images = full_train.data
        raw_labels = full_train.targets

    for idx in range(16):
        sample_table.add_data(wandb.Image(raw_images[idx].numpy()), int(raw_labels[idx]))

    dataset_run.log({"dataset_samples": sample_table})
    dataset_run.log_artifact(dataset_artifact)
    dataset_run.finish()

    return f"mnist-full-batch-{seed}:latest"


class MLP(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 10),
        )

    def forward(self, x):
        return self.net(x)


@torch.no_grad()
def evaluate_full_batch(model, x, y):
    model.eval()
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    acc = (logits.argmax(dim=1) == y).float().mean()
    return loss.item(), acc.item()


def kd_loss(student_logits, teacher_logits, targets, temperature, lambda_kd):
    ce = F.cross_entropy(student_logits, targets)

    log_p_student = F.log_softmax(student_logits / temperature, dim=1)
    p_teacher = F.softmax(teacher_logits / temperature, dim=1)
    kd = F.kl_div(log_p_student, p_teacher, reduction="batchmean") * (temperature ** 2)

    total = (1 - lambda_kd) * ce + lambda_kd * kd
    return total

## Training Functions

Teacher and student training are separated so each experiment run remains easy to read and reuse.

In [5]:
def train_teacher(dataset_artifact_ref):
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        job_type="train-teacher",
        name="teacher-2x1200-relu",
        config={
            "model": "MLP",
            "hidden_size": 1200,
            "num_hidden_layers": 2,
            "activation": "ReLU",
            "epochs": teacher_epochs,
            "lr": teacher_lr,
            "optimizer": "Adam",
            "scheduler": "CosineAnnealingLR",
            "seed": seed,
            "device": str(device),
            "dataset": "MNIST",
            "full_batch": True,
        },
        tags=["teacher", "mnist", "full-batch"],
    )
    run.use_artifact(dataset_artifact_ref)

    teacher = MLP(hidden_size=1200).to(device)
    optimizer = Adam(teacher.parameters(), lr=teacher_lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=teacher_epochs)

    best_state = copy.deepcopy(teacher.state_dict())
    best_val_acc = 0.0
    best_epoch = 0

    print("\nTraining teacher (2x1200 ReLU)...")
    for epoch in range(1, teacher_epochs + 1):
        teacher.train()
        optimizer.zero_grad()

        logits = teacher(x_train)
        loss = F.cross_entropy(logits, y_train)
        loss.backward()
        optimizer.step()

        train_acc = (logits.argmax(dim=1) == y_train).float().mean().item()
        val_loss, val_acc = evaluate_full_batch(teacher, x_val, y_val)
        lr_now = optimizer.param_groups[0]["lr"]

        run.log(
            {
                "step": epoch,
                "lr": lr_now,
                "train/loss": loss.item(),
                "train/acc": train_acc,
                "val/loss": val_loss,
                "val/acc": val_acc,
            },
            step=epoch,
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(teacher.state_dict())
            best_epoch = epoch

        scheduler.step()

        print(
            f"Teacher Epoch {epoch:02d}/{teacher_epochs} | lr={lr_now:.6f} | "
            f"train_loss={loss.item():.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    teacher.load_state_dict(best_state)

    local_ckpt = checkpoints_dir / "teacher_best.pt"
    save_local_checkpoint(
        path=local_ckpt,
        model=teacher,
        optimizer=optimizer,
        epoch=best_epoch,
        best_val_acc=best_val_acc,
        metadata={"run": "teacher", "group": WANDB_GROUP},
    )

    model_artifact = wandb.Artifact(
        name=f"teacher-2x1200-{WANDB_GROUP}",
        type="model",
        description="Best teacher checkpoint by validation accuracy",
        metadata={"best_epoch": best_epoch, "best_val_acc": best_val_acc},
    )
    model_artifact.add_file(str(local_ckpt), name="teacher_best.pt")
    run.log_artifact(model_artifact)

    run.summary["best_epoch"] = best_epoch
    run.summary["best_val_acc"] = best_val_acc
    run.summary["local_checkpoint"] = str(local_ckpt)
    run.finish()

    print(f"Teacher best val acc: {best_val_acc:.4f}")
    return teacher


def train_student(run_name, teacher=None, temperature=1.0, use_kd=False):
    run = wandb.init(
        entity=WANDB_ENTITY,
        project=WANDB_PROJECT,
        group=WANDB_GROUP,
        job_type="train-student",
        name=run_name,
        config={
            "model": "MLP",
            "hidden_size": 300,
            "num_hidden_layers": 2,
            "activation": "ReLU",
            "epochs": student_epochs,
            "lr": student_lr,
            "optimizer": "Adam",
            "scheduler": "CosineAnnealingLR",
            "seed": seed,
            "device": str(device),
            "dataset": "MNIST",
            "full_batch": True,
            "use_kd": use_kd,
            "lambda_kd": lambda_kd if use_kd else 0.0,
            "temperature": temperature if use_kd else 1.0,
        },
        tags=["student", "mnist", "full-batch", "kd" if use_kd else "scratch"],
    )
    run.use_artifact(f"mnist-full-batch-{seed}:latest")

    student = MLP(hidden_size=300).to(device)
    optimizer = Adam(student.parameters(), lr=student_lr)
    scheduler = CosineAnnealingLR(optimizer, T_max=student_epochs)

    history = {
        "step": [],
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "lr": [],
    }

    best_state = copy.deepcopy(student.state_dict())
    best_val_acc = 0.0
    best_epoch = 0

    print(f"\nTraining student: {run_name}")
    for epoch in range(1, student_epochs + 1):
        student.train()
        optimizer.zero_grad()

        student_logits = student(x_train)

        if use_kd:
            with torch.no_grad():
                teacher_logits = teacher(x_train)
            loss = kd_loss(
                student_logits=student_logits,
                teacher_logits=teacher_logits,
                targets=y_train,
                temperature=temperature,
                lambda_kd=lambda_kd,
            )
        else:
            loss = F.cross_entropy(student_logits, y_train)

        loss.backward()
        optimizer.step()

        train_acc = (student_logits.argmax(dim=1) == y_train).float().mean().item()
        val_loss, val_acc = evaluate_full_batch(student, x_val, y_val)
        lr_now = optimizer.param_groups[0]["lr"]

        history["step"].append(epoch)
        history["train_loss"].append(loss.item())
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["lr"].append(lr_now)

        run.log(
            {
                "step": epoch,
                "lr": lr_now,
                "train/loss": loss.item(),
                "train/acc": train_acc,
                "val/loss": val_loss,
                "val/acc": val_acc,
            },
            step=epoch,
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(student.state_dict())
            best_epoch = epoch

        scheduler.step()

        print(
            f"{run_name} | Epoch {epoch:02d}/{student_epochs} | lr={lr_now:.6f} | "
            f"train_loss={loss.item():.4f} train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    student.load_state_dict(best_state)

    ckpt_name = run_name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("=", "")
    local_ckpt = checkpoints_dir / f"{ckpt_name}_best.pt"
    save_local_checkpoint(
        path=local_ckpt,
        model=student,
        optimizer=optimizer,
        epoch=best_epoch,
        best_val_acc=best_val_acc,
        metadata={"run": run_name, "group": WANDB_GROUP, "use_kd": use_kd, "temperature": temperature},
    )

    model_artifact = wandb.Artifact(
        name=f"{ckpt_name}-{WANDB_GROUP}",
        type="model",
        description=f"Best student checkpoint for {run_name}",
        metadata={"best_epoch": best_epoch, "best_val_acc": best_val_acc, "use_kd": use_kd, "temperature": temperature},
    )
    model_artifact.add_file(str(local_ckpt), name=local_ckpt.name)
    run.log_artifact(model_artifact)

    run.summary["best_epoch"] = best_epoch
    run.summary["best_val_acc"] = best_val_acc
    run.summary["local_checkpoint"] = str(local_ckpt)
    run.finish()

    print(f"{run_name} best at epoch {best_epoch} with val_acc={best_val_acc:.4f}")
    return student, history

## Run Experiments

This section logs the dataset artifact, trains the teacher, and then trains the three student variants.

In [6]:
# 1) Log dataset as W&B artifact
mnist_artifact_ref = log_dataset_artifact()

# 2) Train teacher once for distillation
teacher_model = train_teacher(mnist_artifact_ref)
teacher_model.eval()

# 3) Train student from scratch (no KD)
student_scratch, hist_scratch = train_student(
    run_name="Student from scratch",
    teacher=None,
    temperature=1.0,
    use_kd=False,
)

# 4) Train student with KD (lambda=0.5, temperature=1)
student_kd_t1, hist_kd_t1 = train_student(
    run_name="Student KD (lambda=0.5, T=1)",
    teacher=teacher_model,
    temperature=temperatures[0],
    use_kd=True,
)

# 5) Train student with KD (lambda=0.5, temperature=10)
student_kd_t10, hist_kd_t10 = train_student(
    run_name="Student KD (lambda=0.5, T=10)",
    teacher=teacher_model,
    temperature=temperatures[1],
    use_kd=True,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/silvs/.netrc.
wandb: Currently logged in as: gabomfim99 (gabomfim-unicamp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Adding directory to artifact (data/MNIST)... Done. 0.4s



Training teacher (2x1200 ReLU)...
Teacher Epoch 01/15 | lr=0.001000 | train_loss=2.3031 train_acc=0.0949 | val_loss=1.7322 val_acc=0.7380
Teacher Epoch 02/15 | lr=0.000989 | train_loss=1.7316 train_acc=0.7327 | val_loss=1.1459 val_acc=0.7758
Teacher Epoch 03/15 | lr=0.000957 | train_loss=1.1438 train_acc=0.7750 | val_loss=0.7278 val_acc=0.8216
Teacher Epoch 04/15 | lr=0.000905 | train_loss=0.7209 train_acc=0.8265 | val_loss=0.5593 val_acc=0.8318
Teacher Epoch 05/15 | lr=0.000835 | train_loss=0.5498 train_acc=0.8402 | val_loss=0.6990 val_acc=0.7556
Teacher Epoch 06/15 | lr=0.000750 | train_loss=0.6758 train_acc=0.7687 | val_loss=0.5432 val_acc=0.8168
Teacher Epoch 07/15 | lr=0.000655 | train_loss=0.5346 train_acc=0.8187 | val_loss=0.6505 val_acc=0.7910
Teacher Epoch 08/15 | lr=0.000552 | train_loss=0.6372 train_acc=0.7955 | val_loss=0.4427 val_acc=0.8640
Teacher Epoch 09/15 | lr=0.000448 | train_loss=0.4244 train_acc=0.8743 | val_loss=0.4370 val_acc=0.8664
Teacher Epoch 10/15 | lr=0.00

lr,███▇▇▆▆▅▄▃▃▂▂▁▁
step,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train/acc,▁▇▇▇█▇▇▇███████
train/loss,█▆▄▂▂▂▂▂▁▁▁▁▁▁▁
val/acc,▁▃▅▆▂▅▄▇▇▇▇▇███
val/loss,█▅▃▂▃▂▂▁▁▁▁▁▁▁▁
best_epoch,15
best_val_acc,0.8804
local_checkpoint,checkpoints/teacher_...
lr,1e-05
step,15


Teacher best val acc: 0.8804



Training student: Student from scratch
Student from scratch | Epoch 01/20 | lr=0.001000 | train_loss=2.3050 train_acc=0.0878 | val_loss=2.1359 val_acc=0.6332
Student from scratch | Epoch 02/20 | lr=0.000994 | train_loss=2.1379 train_acc=0.6326 | val_loss=1.9608 val_acc=0.7230
Student from scratch | Epoch 03/20 | lr=0.000976 | train_loss=1.9625 train_acc=0.7153 | val_loss=1.7469 val_acc=0.7352
Student from scratch | Epoch 04/20 | lr=0.000946 | train_loss=1.7485 train_acc=0.7327 | val_loss=1.5080 val_acc=0.7324
Student from scratch | Epoch 05/20 | lr=0.000905 | train_loss=1.5100 train_acc=0.7290 | val_loss=1.2763 val_acc=0.7400
Student from scratch | Epoch 06/20 | lr=0.000854 | train_loss=1.2785 train_acc=0.7362 | val_loss=1.0756 val_acc=0.7588
Student from scratch | Epoch 07/20 | lr=0.000794 | train_loss=1.0776 train_acc=0.7619 | val_loss=0.9172 val_acc=0.7910
Student from scratch | Epoch 08/20 | lr=0.000727 | train_loss=0.9175 train_acc=0.7934 | val_loss=0.8018 val_acc=0.8142
Student 

lr,████▇▇▇▆▆▅▄▄▃▃▂▂▂▁▁▁
step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train/acc,▁▆▇▇▇▇▇▇████████████
train/loss,█▇▇▆▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁
val/acc,▁▄▄▄▄▅▆▇▇▇▇▇████████
val/loss,█▇▆▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
best_epoch,19
best_val_acc,0.8552
local_checkpoint,checkpoints/student_...
lr,1e-05
step,20


Student from scratch best at epoch 19 with val_acc=0.8552



Training student: Student KD (lambda=0.5, T=1)
Student KD (lambda=0.5, T=1) | Epoch 01/20 | lr=0.001000 | train_loss=2.1422 train_acc=0.0903 | val_loss=2.1346 val_acc=0.5806
Student KD (lambda=0.5, T=1) | Epoch 02/20 | lr=0.000994 | train_loss=1.9652 train_acc=0.5721 | val_loss=1.9506 val_acc=0.6760
Student KD (lambda=0.5, T=1) | Epoch 03/20 | lr=0.000976 | train_loss=1.7805 train_acc=0.6639 | val_loss=1.7315 val_acc=0.7014
Student KD (lambda=0.5, T=1) | Epoch 04/20 | lr=0.000946 | train_loss=1.5591 train_acc=0.6926 | val_loss=1.4991 val_acc=0.7162
Student KD (lambda=0.5, T=1) | Epoch 05/20 | lr=0.000905 | train_loss=1.3234 train_acc=0.7132 | val_loss=1.2784 val_acc=0.7370
Student KD (lambda=0.5, T=1) | Epoch 06/20 | lr=0.000854 | train_loss=1.0993 train_acc=0.7345 | val_loss=1.0889 val_acc=0.7612
Student KD (lambda=0.5, T=1) | Epoch 07/20 | lr=0.000794 | train_loss=0.9070 train_acc=0.7599 | val_loss=0.9390 val_acc=0.7904
Student KD (lambda=0.5, T=1) | Epoch 08/20 | lr=0.000727 | trai

ValueError: Artifact name may only contain alphanumeric characters, dashes, underscores, and dots. Invalid name: 'student_kd_lambda0.5,_t1-mnist-kd-fullbatch-20260601-232038'

## Visualizations

These plots compare the three student training strategies across steps (epochs).

In [ ]:
# Comparison plots
plt.figure(figsize=(13, 5))

plt.subplot(1, 2, 1)
plt.plot(hist_scratch["step"], hist_scratch["train_loss"], label="Scratch train")
plt.plot(hist_scratch["step"], hist_scratch["val_loss"], label="Scratch val")
plt.plot(hist_kd_t1["step"], hist_kd_t1["train_loss"], label="KD T=1 train")
plt.plot(hist_kd_t1["step"], hist_kd_t1["val_loss"], label="KD T=1 val")
plt.plot(hist_kd_t10["step"], hist_kd_t10["train_loss"], label="KD T=10 train")
plt.plot(hist_kd_t10["step"], hist_kd_t10["val_loss"], label="KD T=10 val")
plt.xlabel("Step (epoch)")
plt.ylabel("Loss")
plt.title("Training and Validation Loss Across Steps")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)

plt.subplot(1, 2, 2)
plt.plot(hist_scratch["step"], hist_scratch["train_acc"], label="Scratch train")
plt.plot(hist_scratch["step"], hist_scratch["val_acc"], label="Scratch val")
plt.plot(hist_kd_t1["step"], hist_kd_t1["train_acc"], label="KD T=1 train")
plt.plot(hist_kd_t1["step"], hist_kd_t1["val_acc"], label="KD T=1 val")
plt.plot(hist_kd_t10["step"], hist_kd_t10["train_acc"], label="KD T=10 train")
plt.plot(hist_kd_t10["step"], hist_kd_t10["val_acc"], label="KD T=10 val")
plt.xlabel("Step (epoch)")
plt.ylabel("Accuracy")
plt.title("Training and Validation Accuracy Across Steps")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(hist_scratch["step"], hist_scratch["train_acc"], label="Scratch")
plt.plot(hist_kd_t1["step"], hist_kd_t1["train_acc"], label="KD T=1")
plt.plot(hist_kd_t10["step"], hist_kd_t10["train_acc"], label="KD T=10")
plt.xlabel("Step (epoch)")
plt.ylabel("Training accuracy")
plt.title("Training Accuracy Comparison Across Steps")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(hist_scratch["step"], hist_scratch["val_acc"], label="Scratch")
plt.plot(hist_kd_t1["step"], hist_kd_t1["val_acc"], label="KD T=1")
plt.plot(hist_kd_t10["step"], hist_kd_t10["val_acc"], label="KD T=10")
plt.xlabel("Step (epoch)")
plt.ylabel("Validation accuracy")
plt.title("Validation Accuracy Comparison Across Steps")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

print("\nLocal checkpoints saved in:", checkpoints_dir.resolve())